In [1]:
import os
import json
import hashlib
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from anytree import Node, RenderTree

warnings.filterwarnings("ignore")

In [2]:
df_train = pd.read_json("icecat_data_train.json")
df_train.head(5)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
1072689,ASUS,,https://images.icecat.biz/img/brand/thumb/161_...,ASUS,https://images.icecat.biz/img/brand/thumb/161_...,K31CD-IT049T,[],153,EN,PCs/Workstations,...,None,None,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...
906402,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,686915-A41,[],2509,EN,Notebook Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...
411281,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,37745,[],953,EN,Fibre Optic Cables,...,None,None,NaN,None,None,None,None,NaN,2833>830>953,Computers & Electronics>Computer Cables>Fibre ...
425903,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,FA889AA#AC3,[],8194,EN,Handheld Mobile Computer Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8194,Computers & Electronics>Computers>Handheld Mob...
1047582,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,109559U,[],153,EN,PCs/Workstations,...,None,None,NaN,None,None,None,"[{'VirtualCategoryID': 194, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...


In [3]:
df_val = pd.read_json("icecat_data_validate.json")
df_val.head(5)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
309544,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,S26361-F5524-L800,[],1563,EN,Internal Solid State Drives,...,None,None,NaN,None,None,None,None,NaN,2833>206>2840>1563,Computers & Electronics>Data Storage>Data Stor...
805366,PanzerGlass,,https://images.icecat.biz/img/brand/thumb/1016...,PanzerGlass,https://images.icecat.biz/img/brand/thumb/1016...,PG1501,[],1568,EN,Screen Protectors,...,None,None,NaN,None,None,None,None,NaN,2833>107>1568,Computers & Electronics>Telecom & Navigation>S...
126809,2-Power,,https://images.icecat.biz/img/brand/thumb/1520...,2-Power,https://images.icecat.biz/img/brand/thumb/1520...,ALT268563B,[],911,EN,Memory Modules,...,None,None,NaN,None,None,None,None,NaN,2833>106>2844>911,Computers & Electronics>Computer Components>Sy...
922232,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,25205062,[],2509,EN,Notebook Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...
567207,Verbatim,,https://images.icecat.biz/img/brand/thumb/669_...,Verbatim,https://images.icecat.biz/img/brand/thumb/669_...,97537,[],194,EN,Keyboards,...,None,None,NaN,None,None,None,None,NaN,2833>191>194,Computers & Electronics>Data Input Devices>Key...


In [4]:
df_test = pd.read_json("icecat_data_test.json")
df_test.head(5)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
500081,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,FSP:G-SW3Z560PRE0S,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...
741063,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,5KM83PA,None,151,EN,Notebooks,...,EN,13,1268560.0,EN,2018-10-21 21:55:51,"[Windows 10 Home 64-bit, Intel® Core™ i7-8565U...","[{'VirtualCategoryID': 329, 'UNCATID': '432115...",NaN,2833>150>151,Computers & Electronics>Computers>Notebooks
1091454,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,4NG33EA,[],153,EN,PCs/Workstations,...,EN,880,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...
522928,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,83061,[],883,EN,Networking Cables,...,None,None,NaN,None,None,None,None,NaN,2833>830>883,Computers & Electronics>Computer Cables>Networ...
479478,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,5PS0A14091,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...


In [5]:
import pandas as pd
import re

TEXT_COLS = [
    "Brand",
    "ProductName",
    "Title",
    "Description.LongProductName",
    "Description.LongDesc",
    "SummaryDescription.LongSummaryDescription",
    "SummaryDescription.ShortSummaryDescription",
    "Category.Name.Value",
    "pathlist_names",
]

def to_text(x):
    if isinstance(x, list):
        x = " ".join(map(str, x))
    if x is None:
        return ""
    x = str(x)
    if x.lower() in ["none", "nan"]:
        return ""
    return x

def build_metadata_text(row):
    parts = [to_text(row.get(col, "")) for col in TEXT_COLS]
    return " ".join(p for p in parts if p)

def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"<[^>]+>", " ", s)              # remove HTML
    s = re.sub(r"[^a-z0-9\-+x/ ]+", " ", s)     # keep limited chars
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [6]:
# take only 1000 rows to work with
df_train = df_train.sample(n=1000, random_state=42).reset_index(drop=True)

#  build metadata_text and metadata_text_clean
df_train["metadata_text"] = df_train.apply(build_metadata_text, axis=1)
df_train["metadata_text_clean"] = df_train["metadata_text"].apply(clean_text)

df_train[["metadata_text", "metadata_text_clean"]].head(2)

,metadata_text,metadata_text_clean
0,Acer 60.SH7N2.001 Acer 60.SH7N2.001 notebook s...,acer 60 sh7n2 001 acer 60 sh7n2 001 notebook s...
1,Tripp Lite Minicom Smart 108 Tripp Lite Minic...,tripp lite minicom smart 108 tripp lite minico...


In [7]:
df_val = df_val.sample(n=1000, random_state=42).reset_index(drop=True)

df_val["metadata_text"] = df_val.apply(build_metadata_text, axis=1)
df_val["metadata_text_clean"] = df_val["metadata_text"].apply(clean_text)

df_val[["metadata_text", "metadata_text_clean"]].head(2)

,metadata_text,metadata_text_clean
0,Lenovo 41Y8342 Lenovo 41Y8342 internal solid s...,lenovo 41y8342 lenovo 41y8342 internal solid s...
1,"HP 15-bs019ni HP 15-bs019ni Red,Black Notebook...",hp 15-bs019ni hp 15-bs019ni red black notebook...


In [8]:
df_test = df_test.sample(n=1000, random_state=42).reset_index(drop=True)

df_test["metadata_text"] = df_test.apply(build_metadata_text, axis=1)
df_test["metadata_text_clean"] = df_test["metadata_text"].apply(clean_text)

df_test[["metadata_text", "metadata_text_clean"]].head(2)

,metadata_text,metadata_text_clean
0,DELL 312-0106 DELL 312-0106 notebook spare par...,dell 312-0106 dell 312-0106 notebook spare par...
1,Xerox PHASER 6250DP ZW-KL LSR 24PPM 256MB 500V...,xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500v...


In [9]:
def make_ft_df(df):
    df_ft = df[["metadata_text_clean", "pathlist_names"]].copy()
    df_ft = df_ft.dropna(subset=["pathlist_names"])
    df_ft = df_ft[df_ft["metadata_text_clean"] != ""]
    df_ft = df_ft.rename(columns={
        "metadata_text_clean": "input_text",
        "pathlist_names": "target_path",
    })
    return df_ft.reset_index(drop=True)

df_train_ft = make_ft_df(df_train)
df_val_ft   = make_ft_df(df_val)
df_test_ft  = make_ft_df(df_test)

df_train_ft.head(3)

,input_text,target_path
0,acer 60 sh7n2 001 acer 60 sh7n2 001 notebook s...,Computers & Electronics>Computers>Notebook Par...
1,tripp lite minicom smart 108 tripp lite minico...,Computers & Electronics>Data Input Devices>KVM...
2,hp 826630-a41 hp 826630-a41 notebook spare par...,Computers & Electronics>Computers>Notebook Par...


In [10]:
df_val_ft.head(3)

,input_text,target_path
0,lenovo 41y8342 lenovo 41y8342 internal solid s...,Computers & Electronics>Data Storage>Data Stor...
1,hp 15-bs019ni hp 15-bs019ni red black notebook...,Computers & Electronics>Computers>Notebooks
2,gigabyte geforce 8400 256mb ddr2 gigabyte gefo...,Computers & Electronics>Computer Components>Sy...


In [11]:
df_test_ft.head(3)

,input_text,target_path
0,dell 312-0106 dell 312-0106 notebook spare par...,Computers & Electronics>Computers>Notebook Par...
1,xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500v...,Computers & Electronics>Printers & Scanners>Pr...
2,startech com 5 ft cat 6 white molded rj45 utp ...,Computers & Electronics>Computer Cables>Networ...


In [12]:
import json
from pathlib import Path

# New clean output folder under /home/jovyan
out_dir = Path("fine_tuning_data")   # this will be /home/jovyan/fine_tuning_data
out_dir.mkdir(parents=True, exist_ok=True)

def make_row(row):
    return {
        "messages": [
            {
                "role": "system",
                "content": "You are an assistant that assigns taxonomy paths to products.",
            },
            {
                "role": "user",
                "content": row["input_text"],
            },
            {
                "role": "assistant",
                "content": row["target_path"],
            },
        ]
    }

for name, df_ft in [("train", df_train_ft), ("val", df_val_ft), ("test", df_test_ft)]:
    out_path = out_dir / f"icecat_{name}_ft.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for _, r in df_ft.iterrows():
            f.write(json.dumps(make_row(r), ensure_ascii=False) + "\n")
    print(f"Saved {len(df_ft)} rows to {out_path}")


Saved 1000 rows to fine_tuning_data/icecat_train_ft.jsonl
Saved 1000 rows to fine_tuning_data/icecat_val_ft.jsonl
Saved 1000 rows to fine_tuning_data/icecat_test_ft.jsonl


In [13]:
file_path = "fine_tuning_data/icecat_train_ft.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline())


{"messages": [{"role": "system", "content": "You are an assistant that assigns taxonomy paths to products."}, {"role": "user", "content": "acer 60 sh7n2 001 acer 60 sh7n2 001 notebook spare part top case top case grey acer 60 sh7n2 001 type top case brand compatibility acer compatibility chromebook c710 acer 60 sh7n2 001 top case acer chromebook c710 notebook spare parts computers electronics computers notebook parts accessories notebook spare parts"}, {"role": "assistant", "content": "Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts"}]}

{"messages": [{"role": "system", "content": "You are an assistant that assigns taxonomy paths to products."}, {"role": "user", "content": "tripp lite minicom smart 108 tripp lite minicom smart 108 kvm switch rack mounting black 100m vga ps/2 x 2 1600 x 1200 cat5 save space and money the smart 108 is a single-user analog cat5 kvm switch that gives you the ability to control multiple computers or servers from a single 

In [14]:
import os
os.getcwd()


'/home/jovyan'

In [15]:
#Load your JSONL into a HuggingFace Dataset

In [16]:
from datasets import load_dataset
import torch

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

train_path = "fine_tuning_data/icecat_train_ft.jsonl"
val_path   = "fine_tuning_data/icecat_val_ft.jsonl"
test_path  = "fine_tuning_data/icecat_test_ft.jsonl"

train_ds = load_dataset("json", data_files=train_path)["train"]
val_ds   = load_dataset("json", data_files=val_path)["train"]
test_ds  = load_dataset("json", data_files=test_path)["train"]

print(train_ds[0])


Device: cuda


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

{'messages': [{'role': 'system', 'content': 'You are an assistant that assigns taxonomy paths to products.'}, {'role': 'user', 'content': 'acer 60 sh7n2 001 acer 60 sh7n2 001 notebook spare part top case top case grey acer 60 sh7n2 001 type top case brand compatibility acer compatibility chromebook c710 acer 60 sh7n2 001 top case acer chromebook c710 notebook spare parts computers electronics computers notebook parts accessories notebook spare parts'}, {'role': 'assistant', 'content': 'Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts'}]}


In [17]:
# Convert messages → text format for LLaMA

In [18]:
def messages_to_text(example):
    s = example["messages"][0]["content"]
    u = example["messages"][1]["content"]
    a = example["messages"][2]["content"]

    return {
        "text": f"<s>[INST] {s}\n{u} [/INST] {a}</s>"
    }

train_ds = train_ds.map(messages_to_text)
val_ds   = val_ds.map(messages_to_text)
test_ds  = test_ds.map(messages_to_text)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [19]:
# we plug in LLaMA-3.1-8B-Instruct (local) + LoRA and train.

In [20]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
import torch

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

# use your local Llama model
model_name = "models/meta-llama/Llama-3.1-8B-Instruct"   # relative to /home/jovyan

# 4-bit quantization so 8B fits on GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

# LoRA adapter (we only train a small part of the model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Device: cuda


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [21]:
# Tokenize datasets for LLaMA

In [22]:
MAX_LEN = 256  

def tokenize_fn(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )
    # labels = input_ids (causal LM learns to predict whole sequence)
    enc["labels"] = enc["input_ids"].copy()
    return enc

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_fn,   batched=True, remove_columns=val_ds.column_names)

train_tok.set_format(type="torch")
val_tok.set_format(type="torch")

print(train_tok[0].keys())


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [23]:
from accelerate import Accelerator

# Save original method
_orig_unwrap_model = Accelerator.unwrap_model

def _patched_unwrap_model(self, model, *args, **kwargs):
    # Drop unexpected kwarg if present
    kwargs.pop("keep_torch_compile", None)
    return _orig_unwrap_model(self, model, *args, **kwargs)

Accelerator.unwrap_model = _patched_unwrap_model

print("✅ Patched Accelerator.unwrap_model to ignore keep_torch_compile")


✅ Patched Accelerator.unwrap_model to ignore keep_torch_compile


In [24]:
# TrainingArguments + Trainer + train

In [25]:
output_dir = "llama_taxonomy_1000"

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=10,           #  will print training loss every 10 steps
    logging_first_step=True,
    report_to="none",
)


In [26]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
)

train_result = trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("LLaMA fine-tuning on 1000 samples finished.")


Step,Training Loss
1,3.245100
10,3.101200
20,2.026000
30,1.444100
40,1.591600
50,1.444800
60,1.387000
70,1.479300
80,1.313900
90,1.127700


LLaMA fine-tuning on 1000 samples finished.


In [27]:
# final train loss
print("Train loss:", train_result.training_loss)

# run validation explicitly
eval_metrics = trainer.evaluate()
print("Eval metrics:", eval_metrics)        # contains 'eval_loss'
print("Validation loss:", eval_metrics["eval_loss"])


Train loss: 1.576269115447998


Eval metrics: {'eval_loss': 1.3376764059066772, 'eval_runtime': 117.5058, 'eval_samples_per_second': 8.51, 'eval_steps_per_second': 4.255, 'epoch': 1.0}
Validation loss: 1.3376764059066772


In [ ]:
# Load fine-tuned LLaMA & evaluate on test set

In [28]:
from transformers import pipeline
import torch

# ---------------------------------------------------------
# 1. Load fine-tuned model + tokenizer (from output_dir)
# ---------------------------------------------------------
model_path = "llama_taxonomy_1000"

gen = pipeline(
    "text-generation",
    model=model_path,
    tokenizer=model_path,
    device=0 if torch.cuda.is_available() else -1
)

# ---------------------------------------------------------
# 2. Helper to extract prediction from output text
# ---------------------------------------------------------
def extract_pred(text):
    if "[/INST]" in text:
        return text.split("[/INST]")[-1].strip()
    return text.strip()

# ---------------------------------------------------------
# 3. Run predictions on all test examples
# ---------------------------------------------------------
preds = []
golds = []

for i in range(len(test_ds)):
    ex = test_ds[i]
    prompt = ex["text"].split("{a}")[0] + ""  # or use full prompt

    out = gen(
        prompt,
        max_new_tokens=50,
        do_sample=False
    )[0]["generated_text"]

    pred = extract_pred(out)
    gold = ex["messages"][2]["content"]

    preds.append(pred)
    golds.append(gold)

# ---------------------------------------------------------
# 4. Compute exact match accuracy
# ---------------------------------------------------------
correct = 0
for p, g in zip(preds, golds):
    if p.strip() == g.strip():
        correct += 1

accuracy = correct / len(test_ds)

print(f"Test Accuracy (exact match): {accuracy*100:.2f}%")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Test Accuracy (exact match): 0.00%


In [29]:

from transformers import pipeline
import torch

# 1. Reload fine-tuned model (if needed)
model_path = "llama_taxonomy_1000"

gen = pipeline(
    "text-generation",
    model=model_path,
    tokenizer=model_path,
    device=0 if torch.cuda.is_available() else -1
)

def get_prediction(example, max_new_tokens=50):
    # Rebuild prompt: system + user, NO gold answer
    s = example["messages"][0]["content"]
    u = example["messages"][1]["content"]
    gold = example["messages"][2]["content"]

    prompt = f"<s>[INST] {s}\n{u} [/INST]"

    out = gen(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )[0]["generated_text"]

    # everything after [/INST] is model’s answer
    if "[/INST]" in out:
        pred = out.split("[/INST]", 1)[1].strip()
    else:
        pred = out.strip()

    return u, gold, pred

# ---- inspect a few examples ----
for i in range(5):   # change 5 -> 10 etc. if you want more
    user_text, gold, pred = get_prediction(test_ds[i])

    print(f"\n=== Example {i} ===")
    print("USER TEXT (truncated):")
    print(user_text[:400], "...")
    print("\nGOLD PATH:")
    print(gold)
    print("\nPREDICTED PATH:")
    print(pred)
    print("-" * 80)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0



=== Example 0 ===
USER TEXT (truncated):
dell 312-0106 dell 312-0106 notebook spare part battery 1900 mah 14 8 v always on the go- no more worries for running out of battery power you can back up your system with this 4-cell lithium-ion primary battery from dell it has an internal circuit board with chips that allow it to communicate with the notebook to monitor battery performance output voltage and temperature it also gives the noteboo ...

GOLD PATH:
Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts

PREDICTED PATH:
Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts</s>
--------------------------------------------------------------------------------

=== Example 1 ===
USER TEXT (truncated):
xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500vl+100 dupl 2400dpi eth xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500vl+100 dupl 2400dpi eth colour 600 x 600 dpi a4 phaser 6250dp a4/legal size color printer 220v 26ppm color/b w 24pp

In [ ]:
# correct = 0
# total = len(test_ds)
# #
# for i in range(total):
#     _, gold, pred = get_prediction(test_ds[i], max_new_tokens=50)
#     if pred.strip() == gold.strip():
#         correct += 1

# print(f"Exact-match accuracy: {correct/total*100:.2f}%")


In [31]:
def get_prediction_fixed(example, max_new_tokens=50):

    full_text = example["text"]

    # extract gold label (after [/INST], before </s>)
    if "[/INST]" in full_text:
        gold = full_text.split("[/INST]")[1]
        gold = gold.replace("</s>", "").strip()
    else:
        gold = ""

    # build prompt: everything before [/INST]
    prompt = full_text.split("[/INST]")[0] + "[/INST]"

    out = gen(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )[0]["generated_text"]

    # extract predicted taxonomy
    if "[/INST]" in out:
        pred = out.split("[/INST]")[1]
    else:
        pred = out

    pred = pred.replace("</s>", "").strip()

    return prompt, gold, pred


In [32]:
correct = 0
total = len(test_ds)

for i in range(total):
    _, gold, pred = get_prediction_fixed(test_ds[i])

    if pred == gold:
        correct += 1

print(f"Exact-match accuracy: {correct/total*100:.2f}%")


Exact-match accuracy: 94.20%


In [33]:
from transformers import pipeline
import torch

# ---------------------------------------------------------
# Load your fine-tuned model
# ---------------------------------------------------------
model_path = "llama_taxonomy_1000"

gen = pipeline(
    "text-generation",
    model=model_path,
    tokenizer=model_path,
    device=0 if torch.cuda.is_available() else -1
)

# ---------------------------------------------------------
# Build the SAME prompt formatting used during training
# ---------------------------------------------------------
def build_prompt(user_text):
    return (
        f"You are an assistant that assigns taxonomy paths to products.\n"
        f"[USER] {user_text}\n"
        f"[ASSISTANT]"
    )

# ---------------------------------------------------------
# Predict taxonomy for a custom user input
# ---------------------------------------------------------
def predict_taxonomy():
    user_input = input("Enter product description:\n\n> ")

    prompt = build_prompt(user_input)

    output = gen(
        prompt,
        max_new_tokens=50,
        do_sample=False
    )[0]["generated_text"]

    # Extract only the assistant’s answer
    pred = output.split("[ASSISTANT]", 1)[-1].strip()

    # Remove extra tokens like </s>
    pred = pred.replace("</s>", "").strip()

    print("\n\n=== Predicted Taxonomy Path ===")
    print(pred)
    print("================================")


# ---------------------------------------------------------
# Run one prediction
# ---------------------------------------------------------
predict_taxonomy()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


Enter product description:

>  hp pavilion 15 intel core i5 8gb ram 512gb ssd windows 11 laptop




=== Predicted Taxonomy Path ===
computers electronics computers laptops notebooks hp pavilion 15 intel core i5 8gb ram 512gb ssd windows 11 laptop computers electronics computers laptops notebooks [/ASSISTANT] Computers & Electronics>Computers>Laptops & Notebooks</Comput


In [34]:
# def extract_clean_taxonomy(generated: str) -> str:
#     """
#     Take the full generated text from LLaMA
#     and return only the clean taxonomy path, e.g.
#     'Computers & Electronics>Computers>Laptops & Notebooks'
#     """

#     text = generated

#     # 1) Drop everything before the last assistant marker, if present
#     for marker in ["[/ASSISTANT]", "[ASSISTANT]", "[/INST]", "</INST>"]:
#         if marker in text:
#             text = text.split(marker, 1)[-1]

#     # 2) Keep only from the last 'Computers & Electronics' onwards
#     anchor = "Computers & Electronics"
#     if anchor in text:
#         text = text[text.rfind(anchor):]

#     # 3) Remove special tokens and HTML-ish stuff
#     text = text.replace("</s>", "")
#     text = text.replace("</", "")
#     text = text.strip()

#     # 4) If there are newlines, keep just the first line
#     text = text.splitlines()[0].strip()

#     return text


In [35]:
# def predict_taxonomy():
#     user_input = input("Enter product description:\n\n> ")

#     prompt = build_prompt(user_input)

#     output = gen(
#         prompt,
#         max_new_tokens=50,
#         do_sample=False
#     )[0]["generated_text"]

#     # use the cleaner
#     pred = extract_clean_taxonomy(output)

#     print("\n\n=== Predicted Taxonomy Path ===")
#     print(pred)
#     print("================================")


In [36]:
import torch; torch.cuda.empty_cache()

In [37]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# -----------------------------
# 1. Load fine-tuned model
# -----------------------------
model_path = "llama_taxonomy_1000"   # folder you saved earlier

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

device = 0 if torch.cuda.is_available() else -1

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

# -----------------------------
# 2. Prompt builder
# -----------------------------
def build_prompt(user_text: str) -> str:
    system = "You are an assistant that assigns taxonomy paths to products."
    return f"{system}\n[USER] {user_text}\n[ASSISTANT]"

# -----------------------------
# 3. Clean taxonomy extractor
# -----------------------------
def extract_clean_taxonomy(generated: str) -> str:
    text = generated

    # a) drop everything before [ASSISTANT]
    if "[ASSISTANT]" in text:
        text = text.split("[ASSISTANT]", 1)[1]

    # b) if model added markers like [/ASSISTANT] or [/INST], keep AFTER them
    for marker in ["[/ASSISTANT]", "[/INST]", "</INST>"]:
        if marker in text:
            text = text.split(marker, 1)[-1]

    # c) keep from the last "Computers & Electronics" onwards (main anchor)
    anchor = "Computers & Electronics"
    if anchor in text:
        text = text[text.rfind(anchor):]

    # d) remove special tokens and trim
    text = text.replace("</s>", "").strip()

    # e) keep only first line
    text = text.splitlines()[0].strip()

    return text

# -----------------------------
# 4. Interactive prediction
# -----------------------------
def predict_taxonomy():
    print("Enter product description (Ctrl+C to stop):\n")
    user_text = input("> ")

    prompt = build_prompt(user_text)

    out = gen(
        prompt,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )[0]["generated_text"]

    pred = extract_clean_taxonomy(out)

    print("\n=== Predicted Taxonomy Path ===")
    print(pred or "(model did not output a clear taxonomy)")
    print("================================")

# -----------------------------
# 5. Run once
# -----------------------------
predict_taxonomy()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


Enter product description (Ctrl+C to stop):



>  hp pavilion 15 intel core i5 8gb ram 512gb ssd windows 11 laptop



=== Predicted Taxonomy Path ===
Computers & Electronics>Computers>Laptops & Notebooks</Computers
